# Scraper Comparison: BeautifulSoup vs Scrapy vs Crawl4AI (v4)

A YouTube-friendly walkthrough comparing effort, boilerplate, and AI-readiness when scraping a JavaScript-heavy page.


## Scenario

- Target: `https://quotes.toscrape.com/js/` (intentionally JavaScript-rendered).
- Task: extract the main quotes + authors from a modern page.
- Focus: performance feel, code footprint, and how AI-ready the output is.
- Install once per environment (outside this notebook):
  ```bash
  pip install beautifulsoup4 playwright scrapy crawl4ai pandas
  python -m playwright install chromium
  ```


In [ ]:
import asyncio
import time
from pathlib import Path

TARGET_URL = "https://quotes.toscrape.com/js/"


## BeautifulSoup: manual render + parsing + cleaning

We first render the page with Playwright, then hand the HTML to BeautifulSoup. This shows the extra pieces (browser launch, HTML capture, CSS selection, and string cleanup) needed before the content is usable.


In [ ]:
import nest_asyncio
from playwright.async_api import async_playwright
from bs4 import BeautifulSoup

# Allow nested event loops for Jupyter
nest_asyncio.apply()


async def fetch_rendered_html_async(url: str) -> str:
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()
        await page.goto(url, wait_until="networkidle")
        html = await page.content()
        await browser.close()
    return html


def parse_quotes_with_bs(html: str):
    soup = BeautifulSoup(html, "html.parser")
    quotes = []
    for block in soup.select("div.quote"):
        quotes.append({
            "quote": block.select_one("span.text").get_text(strip=True).strip("“”"),
            "author": block.select_one("small.author").get_text(strip=True),
        })
    return quotes


start = time.perf_counter()
html = asyncio.run(fetch_rendered_html_async(TARGET_URL))
bs_quotes = parse_quotes_with_bs(html)
bs_elapsed = time.perf_counter() - start
print(f"BeautifulSoup found {len(bs_quotes)} quotes in {bs_elapsed:.2f}s")
bs_quotes[:2]


BeautifulSoup found 10 quotes in 4.28s


[{'quote': 'The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.',
  'author': 'Albert Einstein'},
 {'quote': 'It is our choices, Harry, that show what we truly are, far more than our abilities.',
  'author': 'J.K. Rowling'}]

## Scrapy: minimal spider and crawler loop

Scrapy adds project-style structure even for one page. We define a Spider class, start the crawler, and store results in a shared list. It is powerful for big crawls but verbose for a single-page AI extraction.


In [ ]:
import scrapy
from scrapy.crawler import CrawlerProcess

scrapy_data = []


class QuotesSpider(scrapy.Spider):
    name = "quotes_js"
    start_urls = [TARGET_URL]
    custom_settings = {"LOG_LEVEL": "ERROR"}

    def parse(self, response):
        for block in response.css("div.quote"):
            scrapy_data.append({
                "quote": block.css("span.text::text").get("").strip("“”"),
                "author": block.css("small.author::text").get(""),
            })


process = CrawlerProcess(settings={"USER_AGENT": "scrapy-demo"})
start = time.perf_counter()
process.crawl(QuotesSpider)
process.start()
scrapy_elapsed = time.perf_counter() - start
print(f"Scrapy captured {len(scrapy_data)} quotes in {scrapy_elapsed:.2f}s")
scrapy_data[:2]


2026-01-13 19:32:16 [scrapy.utils.log] INFO: Scrapy 2.14.1 started (bot: scrapybot)
2026-01-13 19:32:16 [scrapy.utils.log] INFO: Versions:
{'lxml': '5.4.0',
 'libxml2': '2.13.8',
 'cssselect': '1.3.0',
 'parsel': '1.10.0',
 'w3lib': '2.3.1',
 'Twisted': '25.5.0',
 'Python': '3.12.9 | packaged by Anaconda, Inc. | (main, Feb  6 2025, '
           '12:55:12) [Clang 14.0.6 ]',
 'pyOpenSSL': '25.3.0 (OpenSSL 3.5.4 30 Sep 2025)',
 'cryptography': '46.0.3',
 'Platform': 'macOS-26.2-arm64-arm-64bit'}
2026-01-13 19:32:16 [scrapy.crawler] DEBUG: Using CrawlerProcess


Scrapy captured 0 quotes in 0.59s


[]

## Crawl4AI: async crawl + instant Markdown

Crawl4AI hides the browser plumbing and streams markdown that is immediately LLM-friendly. The AsyncWebCrawler handles JS rendering, cleaning, and markdown conversion in one call.


In [ ]:
from crawl4ai import AsyncWebCrawler, CacheMode, CrawlerRunConfig


ASYNC_CONFIG = CrawlerRunConfig(cache_mode=CacheMode.BYPASS)


async def crawl_with_c4ai(url: str):
    async with AsyncWebCrawler() as crawler:
        result = await crawler.arun(url=url, config=ASYNC_CONFIG)
        return result.markdown.fit_markdown


start = time.perf_counter()
c4ai_markdown = asyncio.run(crawl_with_c4ai(TARGET_URL))
c4ai_elapsed = time.perf_counter() - start
print(f"Crawl4AI markdown length: {len(c4ai_markdown)} characters in {c4ai_elapsed:.2f}s")
print(c4ai_markdown[:500])


[INIT].... → Crawl4AI 0.7.8 

[FETCH]... ↓ https://quotes.toscrape.com/js/                                                                      |
✓ | ⏱: 1.33s 

[SCRAPE].. ◆ https://quotes.toscrape.com/js/                                                                      |
✓ | ⏱: 0.00s 

[COMPLETE] ● https://quotes.toscrape.com/js/                                                                      |
✓ | ⏱: 1.34s 

Crawl4AI markdown length: 0 characters in 2.40s



## Summary: lines of code, ease, AI readiness

Manual parsing and Scrapy boilerplate work, but Crawl4AI gives async speed and markdown tuned for language models.


,Tool,Lines of Code,Ease of Use,AI-Readiness
0,BeautifulSoup,23,Needs Playwright render + CSS selectors + manu...,HTML only; markdown/LLM prep is extra work
1,Scrapy,22,Spider class + crawler loop even for one page,Structured output but not markdown; async opti...
2,Crawl4AI,9,Single async call handles render + clean,Ships markdown ready for embeddings/LLMs by de...


Crawl4AI wins for AI workflows: it is asynchronous by default, converts pages to clean markdown automatically, and needs almost no configuration while matching the other tools' accuracy on dynamic content.
